# Predicting CONMEBOL World Cup qualifier scores

**DA380A Machine Learning · HT2026 · Group 28** — Lawrence Arciniegas, Gunnar Arias

**Goal:** predict the **home score and the away score** of international football matches from information known *before* kick-off, and evaluate the predictions on the **CONMEBOL World Cup qualifiers** of the 2026 cycle (the 10 South American national teams, 90 matches).

**Dataset:** Mart Jürisoo, *International football results from 1872* (Kaggle `martj42/international-football-results-from-1872-to-2017`; downloaded 13 Oct 2026 from the author's GitHub mirror — see the README).

**Algorithms:** (1) Linear Regression **implemented from scratch** with NumPy · (2) Random Forest Regressor (scikit-learn).

The notebook follows the 11-step structure of the project guidelines. The full plan is in [`docs/PLAN.md`](../docs/PLAN.md).

| Section | Step | Phase |
|---|---|---|
| 1 | Import libraries | 1 |
| 2 | Load the dataset | 1 |
| 3 | Explore the data | 1 |
| 4 | Preprocess the data (feature engineering) | 2 |
| 5 | Define features and target | 2 |
| 6 | Split into training and test sets | 4 |
| 7 | **Algorithm 1 — Linear Regression from scratch** | 3–4 |
| 8 | Algorithm 2 — Random Forest Regressor | 5 |
| 9 | Evaluate both models | 6 |
| 10 | Compare the results | 6 |
| 11 | Visualisations | 6 |

## 1. Import libraries

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import (CONMEBOL, RANDOM_STATE, TARGETS, TEST_END, TEST_START, TRAIN_START,
                      clean_results, load_former_names, load_results)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Load the dataset

`results.csv` has one row per match: date, home and away team, the final score (penalty shoot-outs not included), tournament, city, country, and whether the venue was neutral.

In [ ]:
raw = load_results()
former = load_former_names()
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all").T

## 3. Explore the data

### 3.1 Data quality

In [ ]:
teams = pd.unique(raw[["home_team", "away_team"]].values.ravel())
print(f"Date range: {raw['date'].min().date()} -> {raw['date'].max().date()}")
print(f"Teams: {len(teams)} | tournaments: {raw['tournament'].nunique()}")
print(f"Missing values: {raw.isna().sum().sum()}")
print(f"Fully duplicated rows: {raw.duplicated().sum()}")
print(f"Same date + same teams: {raw.duplicated(subset=['date', 'home_team', 'away_team']).sum()}")
print(f"Negative scores: {((raw['home_score'] < 0) | (raw['away_score'] < 0)).sum()}")
raw[raw.duplicated(subset=["date", "home_team", "away_team"], keep=False)]

In [ ]:
ax = raw.groupby(raw["date"].dt.year).size().plot(figsize=(12, 3.5))
ax.axvspan(TRAIN_START.year, TEST_START.year, color="tab:green", alpha=0.08, label="training window")
ax.axvspan(TEST_START.year, TEST_END.year, color="tab:red", alpha=0.12, label="test window")
ax.set(title="International matches per year", xlabel="", ylabel="Matches")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

The raw data is clean in the technical sense: no missing values, no duplicated rows, no impossible scores. The one pair with the same date and teams (Tahiti vs New Caledonia, 1974) has two *different* scores, so it is most likely two real matches on the same day; it is from before our training window and is kept.

### 3.2 Cleaning: which teams and matches we keep

1. **Former team names.** `former_names.csv` lists 36 old names (e.g. Dahomey → Benin). None of them occurs in `results.csv` — the dataset already uses one name per team — so nothing has to be renamed. We keep the check so it stays visible.
2. **Only FIFA national teams.** The data also contains regional and unrecognised teams (Catalonia, Corsica, Tibet, …) from tournaments such as the Island Games or the CONIFA World Football Cup. They produce extreme results (e.g. 22–0) that have nothing to do with our problem. We keep only teams that have played at least one **World Cup or World Cup qualifier** — a simple, data-driven definition of a FIFA member.
3. **Periods.** The full history is kept, because Phase 2 needs old matches to build team ratings, but every match gets a label: *warm-up* (before 1994, features only), *train* (1994 → Aug 2023), *test* (Sep 2023 → Sep 2025, the 2026 qualifying cycle) and *after*.

In [ ]:
df, notes = clean_results(raw, former)
print("\n".join(notes))

In [ ]:
removed = sorted(set(teams) - set(df["home_team"]) - set(df["away_team"]))
print(f"{len(removed)} teams removed, e.g.:", ", ".join(removed[:15]), "...")
df["period"].value_counts().reindex(["warm-up", "train", "test", "after"]).to_frame("matches")

> **From here on, the exploration only uses the training period** (plus the warm-up history where noted). The test period — including the 90 CONMEBOL qualifiers of the 2026 cycle — is not looked at until the final evaluation, so it can't influence our choices.

In [ ]:
train = df[df["period"] == "train"].copy()
train["total_goals"] = train["home_score"] + train["away_score"]
train["margin"] = train["home_score"] - train["away_score"]
print(f"Training matches: {len(train):,} | teams: {pd.unique(train[['home_team', 'away_team']].values.ravel()).size}")

### 3.3 The targets: home and away goals

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
bins = list(range(0, 10))
labels = [str(b) for b in bins[:-1]] + ["9+"]
counts = pd.DataFrame({
    side: train[col].clip(upper=9).value_counts(normalize=True).reindex(bins, fill_value=0).values
    for side, col in [("Home goals", "home_score"), ("Away goals", "away_score")]
}, index=labels)
counts.plot(kind="bar", ax=ax, rot=0, width=0.8)
ax.set(title="Goals per team in a match (training period)", xlabel="Goals", ylabel="Share of matches")
plt.tight_layout(); plt.show()

train[TARGETS].agg(["mean", "var", "median", "max"]).T.round(2)

In [ ]:
print("Matches won by 10 or more goals:", (train["margin"].abs() >= 10).sum())
train.loc[train["margin"].abs().sort_values(ascending=False).index[:5],
          ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]]

### 3.4 Home advantage

In [ ]:
def outcome_table(d):
    return pd.Series({
        "matches": len(d),
        "home goals": d["home_score"].mean(),
        "away goals": d["away_score"].mean(),
        "home win %": 100 * (d["margin"] > 0).mean(),
        "draw %": 100 * (d["margin"] == 0).mean(),
        "away win %": 100 * (d["margin"] < 0).mean(),
    })

venue = train.groupby(train["neutral"].map({False: "home venue", True: "neutral venue"})).apply(outcome_table)
ax = venue[["home win %", "draw %", "away win %"]].plot(kind="bar", rot=0, figsize=(7, 3.5))
ax.set(title="Results by venue (training period)", xlabel="", ylabel="% of matches")
plt.tight_layout(); plt.show()
venue.round(2)

### 3.5 Match importance

In [ ]:
importance = train.groupby("tournament_group", observed=True).apply(
    lambda d: pd.Series({"matches": len(d), "goals per match": d["total_goals"].mean(),
                         "mean |margin|": d["margin"].abs().mean(),
                         "home win %": 100 * (d["margin"] > 0).mean()}))
importance.round(2)

### 3.6 CONMEBOL World Cup qualifiers (training period)

Since the qualifiers for the 1998 World Cup, CONMEBOL plays one league: every team meets every other team home and away.

In [ ]:
wcq = train[train["is_conmebol_wcq"] & (train["date"] >= "1996-01-01")].copy()
cycle_end = {1997: 1998, 2001: 2002, 2005: 2006, 2009: 2010, 2013: 2014, 2017: 2018, 2022: 2022}
wcq["cycle"] = wcq["date"].dt.year.map(lambda y: next(wc for end, wc in cycle_end.items() if y <= end))
print(wcq.groupby("cycle").size().rename("matches").to_string())
print(f"\nAll CONMEBOL qualifiers 1996-2022: {len(wcq)}")
outcome_table(wcq).round(2)

In [ ]:
rows = []
for team in CONMEBOL:
    home = wcq[wcq["home_team"] == team]
    away = wcq[wcq["away_team"] == team]
    home_pts = 3 * (home["margin"] > 0) + (home["margin"] == 0)
    away_pts = 3 * (away["margin"] < 0) + (away["margin"] == 0)
    rows.append({"team": team, "played": len(home) + len(away),
                 "points per game, home": home_pts.mean(), "points per game, away": away_pts.mean(),
                 "goals for per game": (home["home_score"].sum() + away["away_score"].sum()) / (len(home) + len(away)),
                 "goals against per game": (home["away_score"].sum() + away["home_score"].sum()) / (len(home) + len(away))})
teams_wcq = pd.DataFrame(rows).set_index("team").sort_values("points per game, home", ascending=False)

ax = teams_wcq[["points per game, home", "points per game, away"]].plot(kind="bar", rot=0, figsize=(11, 3.8))
ax.set(title="CONMEBOL qualifiers 1996-2022: points per game at home and away", xlabel="", ylabel="Points per game")
plt.tight_layout(); plt.show()
teams_wcq.round(2)

### 3.7 Altitude

Bolivia plays its home qualifiers in La Paz (≈ 3,640 m) and recently El Alto (≈ 4,100 m); Ecuador mostly in Quito (≈ 2,850 m); Colombia sometimes in Bogotá (≈ 2,640 m). Do visiting teams struggle there?

In [ ]:
HIGH_ALTITUDE_M = {"La Paz": 3640, "El Alto": 4100, "Quito": 2850, "Bogotá": 2640}
wcq["venue"] = np.where(wcq["city"].isin(["La Paz", "El Alto"]), "Bolivia: La Paz / El Alto",
               np.where(wcq["city"].isin(["Quito", "Bogotá"]), "Quito / Bogotá", "other venues"))
altitude = wcq.groupby("venue").apply(outcome_table)
altitude.round(2)

In [ ]:
# Same team, different altitude: Bolivia and Ecuador at home vs away
for team in ["Bolivia", "Ecuador"]:
    print(f"{team}: points per game at home {teams_wcq.loc[team, 'points per game, home']:.2f}, "
          f"away {teams_wcq.loc[team, 'points per game, away']:.2f}")

### 3.8 Main observations

- **Data quality.** 49,945 matches from 1872 to Oct 2026 with no missing values, no duplicated rows and no impossible scores. Removing regional and unrecognised teams dropped 2,565 matches; the training period then has **25,266 matches between 211 national teams**. The dip in 2020 is the COVID-19 pandemic.
- **The targets are small, skewed counts.** Home teams score 1.62 goals on average and away teams 1.08, with a median of 1 for both. 26% of home teams and 40% of away teams score 0. The variance (2.73 and 1.65) is larger than the mean — a long tail of big wins. 101 training matches were won by 10 or more goals (e.g. Australia 31–0 American Samoa); they are real, official matches, so we keep them, but they will push up the RMSE of every model. Team-strength features should explain most of them.
- **Home advantage is large.** At a home venue the home team scores 1.67 goals against 1.00 and wins 50.8% of the matches (away wins 25.7%). On neutral ground the gap almost disappears (1.49 vs 1.29). → `neutral` must be a feature.
- **Match importance matters.** Qualifiers have the most lopsided results (mean margin 1.84 goals, against 1.37 in friendlies), because they include mismatches between strong and very weak teams. → a match-importance feature.
- **CONMEBOL qualifiers 1996–2022:** 593 matches in 7 cycles (72 when Brazil qualified automatically for 1998 and as hosts for 2014; 89 for 2022 because Brazil vs Argentina in September 2021 was suspended). The home team wins 54.0% (draw 24.3%, away win 21.7%) and scores 1.67 goals against 0.95 — a **stronger home advantage than in international football overall**.
- **Brazil and Argentina** are the strongest teams both at home and away (2.59 / 1.56 and 2.30 / 1.57 points per game).
- **Altitude.** Bolivia takes 1.59 points per game at home but only 0.13 away; Ecuador 2.13 against 0.72 — by far the biggest home/away gaps. In Quito and Bogotá the home team wins 59.7% and visitors score only 0.72 goals per game. In La Paz / El Alto the home win rate is lower (42.6%) because Bolivia is a weaker team overall, but compared with Bolivia's away record the effect is huge. → we move the **altitude** feature from "stretch" to the planned features of Phase 2.

**What this means for Phase 2:** team strength (Elo ratings) to explain the big wins and the dominance of Brazil and Argentina, recent form, the `neutral` flag, a match-importance feature, and venue altitude.

## 4. Preprocess the data — feature engineering

_To do — Phase 2: Elo ratings, recent form, head-to-head, venue, match importance (`src/features.py`)._

## 5. Define features and target

_To do — Phase 2._

## 6. Split into training and test sets

_To do — Phase 4: chronological split (train < 1 Sep 2023 ≤ test)._

## 7. Algorithm 1 — Linear Regression from scratch

_To do — Phases 3–4: `LinearRegressionScratch` (normal equation + gradient descent + L2)._

## 8. Algorithm 2 — Random Forest Regressor

_To do — Phase 5: grid search with `TimeSeriesSplit`._

## 9. Evaluate both models

_To do — Phase 6: MAE, MSE, RMSE, R² against the baselines._

## 10. Compare the results

_To do — Phase 6._

## 11. Visualisations

_To do — Phase 6._